In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [11]:
df = pd.read_csv("medical_data.csv")

df.head()

,Age,Gender,Systolic_BP,Cholesterol,Blood_Sugar,BMI,Smoking,Exercise_Hours,Family_History,Disease
0,58,1,165,236,92,31.9,1,7.6,1,0
1,71,0,91,257,240,29.9,1,2.8,0,1
2,48,1,164,324,71,19.2,1,8.5,0,0
3,34,1,117,212,235,34.3,1,0.6,1,1
4,62,1,171,267,226,21.8,0,9.5,0,1


In [12]:
X = df.drop("Disease", axis=1)
y = df["Disease"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (5000, 9)
Target Shape: (5000,)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [14]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:
lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict(X_test_scaled)

lr_acc = accuracy_score(y_test, lr_pred)

print("Logistic Regression Accuracy:", round(lr_acc, 4))

Logistic Regression Accuracy: 0.786


In [16]:
svm = SVC(
    kernel="rbf",
    random_state=42
)

svm.fit(X_train_scaled, y_train)

svm_pred = svm.predict(X_test_scaled)

svm_acc = accuracy_score(y_test, svm_pred)

print("SVM Accuracy:", round(svm_acc, 4))

SVM Accuracy: 0.787


In [17]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", round(rf_acc, 4))

Random Forest Accuracy: 0.782


In [18]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Support Vector Machine",
        "Random Forest"
    ],
    "Accuracy": [
        lr_acc,
        svm_acc,
        rf_acc
    ]
})

results = results.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

results

,Model,Accuracy
0,Support Vector Machine,0.787
1,Logistic Regression,0.786
2,Random Forest,0.782


In [19]:
best_model_name = results.loc[0, "Model"]

print("Best Model:", best_model_name)

Best Model: Support Vector Machine


In [20]:
if best_model_name == "Logistic Regression":
    y_pred = lr_pred

elif best_model_name == "Support Vector Machine":
    y_pred = svm_pred

else:
    y_pred = rf_pred

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Confusion Matrix
[[512  88]
 [125 275]]

Classification Report
              precision    recall  f1-score   support

           0       0.80      0.85      0.83       600
           1       0.76      0.69      0.72       400

    accuracy                           0.79      1000
   macro avg       0.78      0.77      0.77      1000
weighted avg       0.79      0.79      0.79      1000



In [21]:
import joblib

if best_model_name == "Logistic Regression":
    best_model = lr

elif best_model_name == "Support Vector Machine":
    best_model = svm

else:
    best_model = rf

joblib.dump(best_model, "disease_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Model Saved Successfully")

Model Saved Successfully
